# Notebook 05 - Compare Base / Raw SFT / Adapted SFT

This notebook creates a qualitative comparison table for the three model conditions.

The important experimental discipline: use the same prompts and sampling settings for all three models.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from kirundi_sft_starter.utils import load_yaml, read_jsonl

config = load_yaml(ROOT / "configs/project.yaml")
config["models"]["registry"]


## Generate or load responses

Use the CLI to generate comparable outputs. Add `--dry-run` if you want placeholder files while reading the notebook.

```bash
python scripts/generate_model_responses.py --model-key base
python scripts/generate_model_responses.py --model-key sft_raw
python scripts/generate_model_responses.py --model-key sft_adapted
```

In [ ]:
frames = []
for model_key, model_cfg in config["models"]["registry"].items():
    path = ROOT / model_cfg["response_path"]
    if path.exists():
        df = pd.DataFrame(read_jsonl(path))
        df["model"] = model_key
        frames.append(df)

if frames:
    responses = pd.concat(frames, ignore_index=True)
    display(responses.head())
else:
    print("No response files yet. Run the generation commands above.")

In [ ]:
if frames:
    wide = responses.pivot_table(index=["prompt_id", "prompt"], columns="model", values="response", aggfunc="first").reset_index()
    display(wide)

    length_summary = responses.assign(response_chars=responses["response"].str.len()).groupby("model")["response_chars"].mean().reset_index()
    display(length_summary)

## Qualitative review questions

- Did the answer stay in Kirundi/Rundi?
- Did it answer the question directly?
- Did it copy unwanted reasoning traces?
- Did Adaption appear to improve clarity or formatting?
- Are there examples that need native speaker review before any conclusion?